In [1]:
%load_ext autoreload
%autoreload 2

# Define autroreload so that it doesn't cause pain in the ass when we change the functions and run this notebook

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parents[2]
sys.path.append(str(project_root))

print(project_root)

from defs.diffusion.diffusion import *
from defs.diffusion.epsilon import *
from defs.diffusion.training_defs import *
from defs.diffusion.loss import *
from defs.diffusion.noise_scheduling import *

import torch.optim as optim

C:\SenkDosya\Projects\FINCH-Science_SyntheticData


In [4]:
# Predefine all the necessary inputs of the training function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Loss:
loss = loss_mse_sam(0.01)

# Epsilon:
cfg_model = {
    'time_embed': {
        'hidden_dim': 64,
        'hidden_n': 5
    },
    'ab_embed': {
        'hidden_dim': 128,
        'hidden_n': 4,
        'ab_dim': 3
    },
    'denoiser': {
        'hidden_dim': 256,
        'hidden_n': 8,
        'spec_dim': 81
    }
}
epsilon= Epsilon_MLP(cfg_model= cfg_model)
epsilon = epsilon.to(device)

# Scheduler:
scheduler = CosSchedule(2000)

# DDPM:
ddpm = cond_diffusion(epsilon=epsilon, scheduler=scheduler)

# Optimizer:
optimizer = optim.Adam(epsilon.parameters(), lr=1e-2)

# Data handle:
data_handle = str(project_root)+'\data\simpler_data_rwc.csv'

# cfg_train:
cfg_train = {
    'cfg_loader': {
        'test': 23, 't_batch': 1, 'validate': 4, 'epoch': 50
    },
    'range': [900, 1700],
    'device': device
}

<>:39: SyntaxWarning: invalid escape sequence '\d'
<>:39: SyntaxWarning: invalid escape sequence '\d'


cuda


C:\Users\Ege Artan\AppData\Local\Temp\ipykernel_21324\1828605018.py:39: SyntaxWarning: invalid escape sequence '\d'
  data_handle = str(project_root)+'\data\simpler_data_rwc.csv'


In [5]:
print('N of params in epsillon: ', get_n_params(epsilon))

N of params in epsillon:  725457


In [6]:
collector_dict, ds_train, ds_validation, ds_test, ddpm = train_diffusion(
    cfg_train= cfg_train, cond_diffusion= ddpm, loss= loss, optimizer= optimizer, data_handle= data_handle 
)

tensor(1.3925, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.8074, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.4695, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.0472, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.2315, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(63.8755, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.0117, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.9365, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.0912, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(3.6460, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.1341, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  i

In [1]:
import matplotlib.pyplot as plt

spectrum = collector_dict['gen_spec']['validate'][50]['generated'][3]

wavelengths = np.arange(spectrum.shape[0])

plt.plot(wavelengths, spectrum)
plt.show()


NameError: name 'collector_dict' is not defined

This shows our results are almost complete bullshit. The main perpetrator is the loss function, it is very, very interesting that the loss is almost always around 1...

In [ ]:
train_loss = collector_dict['losses']['train']
test_loss = collector_dict['losses']['test']
val_loss = collector_dict['losses']['val']



torch.Size([23])


Next things to do:

Look at which part of the loss are going crazy.